# Suite CPD — The C Preprocessor dialect

`Expressions.C.Preprocessor` models the preprocessing directives C and C++ share, each standard defining its own
(ISO/IEC 9899, 6.10; ISO/IEC 14882, [cpp]), `defined(name)` within their conditions, and the comments kept among them,
as mbse-programs has them: one set of directives for both languages, the C family's dialect that C and C++ read. A
condition is a C or a C++ term, `defined(name)` one of C's or C++'s; a directive and a comment are an enumeration's items
in C Types' and Cpp Types' terms. Its kinds are validated, bind nothing but a conditional directive's macros' names
within its condition, cover C's and C++'s editions, print as mbse-programs prints them, and are refused by evaluation by
name.

In [ ]:
import { Framework } from "@mbse/expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as CFamily from "@mbse/expressions/C/Expressions";
import * as CPreprocessor from "@mbse/expressions/C/Preprocessor";
import * as CTypes from "@mbse/expressions/C/Types";
import * as CppFamily from "@mbse/expressions/Cpp/Expressions";
import * as CppTypes from "@mbse/expressions/Cpp/Types";
import * as AMD64 from "@mbse/expressions/SystemV/AMD64";
import { Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Validators } from "@mbse/schemas/Framework";
import { assert, equal, raises, same } from "./support.js";

const { NotImplementedError, ValueError } = Errors;
const [C, CV, CText] = [CFamily.Expressions, CFamily.Evaluators, CFamily.Text];
const [PP, PPText, CT, XT] = [CPreprocessor.Expressions, CPreprocessor.Text, CTypes.Expressions, CppTypes.Expressions];
const [CPP, XV, XText] = [CppFamily.Expressions, CppFamily.Evaluators, CppFamily.Text];

const store: any = new Builders.Store(); // the C family's dialects, the preprocessor's install before the others': in any order
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, CPreprocessor.install, CFamily.install, CTypes.install,
  CppFamily.install, CppTypes.install, AMD64.install]) {
  store.transaction(install);
}
for (const define of [CFamily.define, CppFamily.define, AMD64.define, CPreprocessor.define]) store.transaction(define); // C's and C++'s types, its editions
const AMD64_ = store.Instances.Expressions.SystemV.AMD64; // the ABI a holder names, whose types C and C++ leave to it

/** The term value `spec` writes, the root of `dialect`'s holder, naming x86-64's ABI, written in a transaction. */
function term(spec: unknown, dialect: F.Dialect = PP.DIALECT): any {
  return F.term(store.transaction(() => dialect.path.split(".").reduce((at: any, part) => at[part], store.Expressions)
    .Expression((e: any) => e.root(dialect.spec(spec)).abi([AMD64_]))));
}

const validate = (spec: unknown, bound: string[] = [], dialect: F.Dialect = PP.DIALECT): string[] =>
  dialect.validate(term(spec, dialect), bound);
const text = (spec: unknown, dialect: F.Dialect = PP.DIALECT): string => PPText.ToText(term(spec, dialect));
const union = (name: string): string[] => store.registered(name).branches.map((b: any) => b.name); // a union's branches
const schemas = (name: string): Map<string, any> => new Map(store.registered(name).branches.map((b: any) => [b.name, b.schema]));
const spelled = (word: string) => (i: any) => i.spelling(word); // an identifier, of its spelling
const own = (spec: unknown) => PP.DIALECT.spec(spec); // a term of its own, where any dialect's may stand
const defined = (word: string, dialect: F.Dialect = C.DIALECT) => dialect.spec((u: any) => u.definedexpression((d: any) => d.name(spelled(word))));
const enumerator = (word: string, dialect: F.Dialect = CT.DIALECT) => dialect.spec((u: any) => u.enumerator((r: any) => r.name(spelled(word))));
const comment = (words: string, block = false) => (u: any) => u.comment((c: any) => c.text(words).block(block));
const condition = C.binary("&&", defined("X"), C.binary(">", C.identifier("X"), 1n)); // defined(X) && X > 1, a C condition
const else_ = (items: unknown[]) => (a: any) => a.elsedirective((e: any) => e.items(items));
const if_ = (u: any) => u.ifdirective((d: any) => d.condition(condition).items([enumerator("A")]).alternative(
  (a: any) => a.elifdefdirective((e: any) => e.negated(true).name(spelled("Z")).items([enumerator("B")]).alternative(
    else_([own(comment(" none"))])))));


<a id="CPD-01"></a>
## CPD-01 · Its union and holder: its kinds and invalid syntax, definedexpression in C's and C++'s terms, the directives and comment in C Types' and Cpp Types'; a holder refining the framework's, round-tripped through JSON

In [ ]:
{
  assert(same(PP.TAGS, ["includedirective", "definedirective", "ifdirective", "ifdefdirective", "elifdirective", "elifdefdirective",
    "elsedirective", "otherdirective", "definedexpression", "comment"]) && same(PP.DIRECTIVES, PP.TAGS.slice(0, 8)));
  assert(same(union(PP.DIALECT.union), [...PP.TAGS, "error", "missing"]) && same([...PP.DIALECT.kinds().keys()], [...PP.TAGS, "error", "missing"]));
  assert(PP.HELPERS.length === 0 && PP.DIALECT.name() === "C Preprocessor" && PP.DIALECT.tag === "c_preprocessor");
  assert(same(PP.DIALECT.reads(), [C.DIALECT, CT.DIALECT, CPP.DIALECT, XT.DIALECT])); // C's and C++'s expressions and types
  const kinds = schemas(PP.DIALECT.union);
  for (const dialect of [C.DIALECT, CPP.DIALECT]) { // a condition's term, in C's and C++'s (first, as its install ran first)
    assert(union(dialect.union).includes("definedexpression"));
    assert(schemas(dialect.union).get("definedexpression") === kinds.get("definedexpression")
      && dialect.kindNamed("definedexpression") === PP.DIALECT.kinds().get("definedexpression"));
  }
  for (const dialect of [CT.DIALECT, XT.DIALECT]) { // an enumeration's items, in C Types' and Cpp Types'
    assert(same(union(dialect.union).slice(0, 9), [...PP.DIRECTIVES, "comment"]) && PP.DIRECTIVES.every((tag) => schemas(dialect.union).get(tag) === kinds.get(tag)));
  }
  assert(!union(C.DIALECT.union).includes("comment") && !union(CPP.DIALECT.union).includes("includedirective")); // no expression holds one
  const Holder: any = store.registered("Expressions.C.Preprocessor.Expression");
  assert(Holder.ref && Holder.refines === store.registered("Expressions.Framework.Expression") && F.isExpression(Holder));
  assert(Holder.properties.get("root").schema === store.registered(PP.DIALECT.union) && same([...Holder.adjacencies.keys()], ["abi"]));
  const [holder, linked] = store.transaction(() => {
    const holder = store.Expressions.C.Preprocessor.Expression((e: any) => e.root(PP.DIALECT.spec(if_)).abi([AMD64_]));
    const linked = store.Schemas.OfExpression((s: any) => s.name("cpd.if").expression((a: any) => a.insert((e: any) => e.expression(holder))));
    return [holder, linked];
  });
  assert(Validators.Validate(store)(Holder, holder).length === 0 && PP.DIALECT.validate(holder).length === 0);
  assert(F.dialectOf(F.term(holder)) === PP.DIALECT.union && F.kindOf(F.term(holder))[0] === "ifdirective");
  assert(F.same((JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, holder)) as any).root, holder.root));
  assert(same([...(new Builders.OfExpression.Builder(linked, false, store) as any).expression].map((entry: any) => F.same(entry.expression, holder)), [true]));
  raises(Error, () => store.transaction(() => // its root holds its own terms
    store.Expressions.C.Preprocessor.Expression((e: any) => e.root(PP.DIALECT.spec(C.binary("+", C.identifier("x"), 1n))))),
  "'binaryexpression' is not a member of a union value");
}


<a id="CPD-02"></a>
## CPD-02 · Its kinds: mbse-programs' properties in its order, a condition, a macro and the items any dialect's terms, a branch after another its own, a name C's identifier

In [ ]:
{
  const props = (name: string): string[] => [...store.registered(`Expressions.C.Preprocessor.${name}`).properties.keys()];
  assert(same(props("IncludeDirective"), ["directive", "path", "system", "macro"]));
  assert(same(props("DefineDirective"), ["name", "function_like", "parameters", "variadic", "replacement"]));
  assert(same(props("IfDirective"), ["condition", "items", "alternative"]) && same(props("ElifDirective"), ["condition", "items", "alternative"]));
  assert(same(props("IfdefDirective"), ["negated", "name", "items", "alternative"]) && same(props("ElifdefDirective"), props("IfdefDirective")));
  assert(same(props("ElseDirective"), ["items"]) && same(props("OtherDirective"), ["directive", "text"]));
  assert(same(props("DefinedExpression"), ["name"]) && same(props("Comment"), ["block", "text", "trailing"]));
  const prop = (name: string, member: string): any => store.registered(`Expressions.C.Preprocessor.${name}`).properties.get(member).schema;
  const [Term, Identifier] = [store.registered("Expressions.Framework.Term"), store.registered("Expressions.C.Expressions.Identifier")];
  assert(prop("IfDirective", "condition") === Term && prop("IncludeDirective", "macro") === Term && prop("IfDirective", "items").element === Term);
  assert(prop("IfDirective", "alternative") === store.registered(PP.DIALECT.union) && prop("ElifdefDirective", "alternative") === store.registered(PP.DIALECT.union));
  assert(prop("DefinedExpression", "name") === Identifier && prop("IfdefDirective", "name") === Identifier && prop("DefineDirective", "parameters").element === Identifier);
  // As forms: a directive's condition and its branch after it are slots, its items its arguments after them
  const made = F.form(term(if_));
  assert(made.kind === "ifdirective" && same(made.arguments.map((a: any) => F.kindOf(a)[0]), ["binaryexpression", "elifdefdirective", "enumerator"]));
  assert(same(made.arguments.map((a: any) => F.dialectOf(a)), [C.DIALECT.union, PP.DIALECT.union, CT.DIALECT.union]));
  assert(F.same(term(PP.DIALECT.make(made)), term(if_)));
}


<a id="CPD-03"></a>
## CPD-03 · Its checks: what each kind requires, an include's path or macro and its directive, a branch after another, an enumeration's items, a name's spelling, with mbse-programs' messages

In [ ]:
{
  const cases: [unknown, string[]][] = [
    [(u: any) => u.includedirective((d: any) => d), ["an IncludeDirective has a path or a macro"]],
    [(u: any) => u.includedirective((d: any) => d.path("a.h").macro(C.identifier("H"))), ["an IncludeDirective has a path or a macro"]],
    [(u: any) => u.includedirective((d: any) => d.macro(C.identifier("H"))), []], // a macro's name, bound within it
    [(u: any) => u.includedirective((d: any) => d.directive("require").path("a.h")), ["IncludeDirective.directive cannot be 'require'"]],
    [(u: any) => u.includedirective((d: any) => d.directive("include_next").path("a.h")), []],
    [(u: any) => u.definedirective((d: any) => d), ["a definedirective needs a name"]],
    [(u: any) => u.definedirective((d: any) => d.name(spelled("F")).function_like(true).parameters([spelled("1a")])), ["'1a' is not an identifier"]],
    [(u: any) => u.ifdirective((d: any) => d), ["an ifdirective needs a condition"]],
    [(u: any) => u.ifdefdirective((d: any) => d), ["an ifdefdirective needs a name"]],
    [(u: any) => u.elifdirective((d: any) => d), ["an elifdirective needs a condition"]],
    [(u: any) => u.elifdefdirective((d: any) => d.name(spelled("2x"))), ["'2x' is not an identifier"]],
    [(u: any) => u.definedexpression((d: any) => d), ["a definedexpression needs a name"]],
    [(u: any) => u.ifdefdirective((d: any) => d.name(spelled("A")).alternative((a: any) => a.ifdefdirective((e: any) => e.name(spelled("B"))))),
      ["an IfdefDirective's alternative is an elifdirective, an elifdefdirective or an elsedirective"]],
    [(u: any) => u.ifdirective((d: any) => d.condition(C.literal(1n)).items([C.literal(2n)])), ["an IfDirective's items are enumerators, directives and comments"]],
    [(u: any) => u.elsedirective((d: any) => d.items([own(comment(" c")), enumerator("A", XT.DIALECT)])), []], // any language's, from its holder
    [(u: any) => u.otherdirective((d: any) => d), []], [(u: any) => u.comment((c: any) => c), []], // the null directive, an empty comment
  ];
  for (const [spec, problems] of cases) assert(same(validate(spec), problems), `${problems} != ${validate(spec)}`);
  assert(same(validate((u: any) => u.ifdirective((d: any) => d.condition((c: any) => c.c((v: any) => v.literal((li: any) => li.value((n: any) => n.as_scalar(1n)).suffix("q")))))), [
    "condition: a literal's suffix cannot be 'q'"])); // a C condition's own checks
}


<a id="CPD-04"></a>
## CPD-04 · What it binds: a conditional directive's macros' names, within its condition alone; defined's name is a macro's, not a reference; nothing refines Let

In [ ]:
{
  assert(validate((u: any) => u.ifdirective((d: any) => d.condition(C.binary(">", C.identifier("X"), 1n)))).length === 0); // X names a macro
  assert(validate((u: any) => u.ifdirective((d: any) => d.condition(CPP.binary(">", CPP.identifier("X"), 1n)))).length === 0); // C++'s too
  assert(same([...PP.macros(term(condition, C.DIALECT))], ["X"]) && PP.macros(null).size === 0);
  assert(SY.free(term(if_), PP.DIALECT).size === 0 && same([...SY.free(term(condition, C.DIALECT), C.DIALECT)], ["X"])); // bound by its directive alone
  assert(C.DIALECT.validate(term(defined("X"), C.DIALECT)).length === 0 && SY.free(term(defined("X"), C.DIALECT), C.DIALECT).size === 0);
  const valued = (u: any) => u.ifdirective((d: any) => d.condition(C.identifier("X")).items([enumerator("A", CT.DIALECT)]).alternative(
    else_([CT.DIALECT.spec((e: any) => e.enumerator((r: any) => r.name(spelled("B")).value(C.identifier("X"))))])));
  assert(same(validate(valued), ["alternative: argument 0: value: variable 'X' is not bound"])); // within its condition alone
  assert(!store.registered(PP.DIALECT.union).branches.some((b: any) => F.isBinder(b.schema)));
}


<a id="CPD-05"></a>
## CPD-05 · Among the family's: defined(name) in C's and C++'s conditions, the directives and comments among C's and C++'s enumerators, each validated and printed from its family's holders

In [ ]:
{
  // defined(name), a C term or a C++ term, within an expression
  for (const [dialect, printer] of [[C.DIALECT, CText.ToText], [CPP.DIALECT, XText.ToText]] as [F.Dialect, (e: unknown) => string][]) {
    const either = dialect.spec((u: any) => u.binaryexpression((b: any) => b.left(defined("A", dialect)).operator("||").right(defined("B", dialect))));
    assert(dialect.validate(term(either, dialect)).length === 0 && printer(term(either, dialect)) === "defined(A) || defined(B)");
    assert(F.kindOf(term(defined("A", dialect), dialect))[0] === "definedexpression");
  }
  // Directives and comments among an enumeration's enumerators, in C's and C++'s, from C's and C++'s holders
  const enumeration = (types: F.Dialect, items: unknown[]) => types.spec((t: any) => t.typeid((i: any) => i.specifiers([(s: any) => s.enumspecifier(
    (e: any) => e.key("enum").name((n: any) => n.identifier(spelled("E"))).body(true).enumerators(items))])));
  for (const [types, family, printer] of [[CT.DIALECT, C, CText.ToText], [XT.DIALECT, CPP, XText.ToText]] as [F.Dialect, any, (e: unknown) => string][]) {
    const body = enumeration(types, [enumerator("A", types), types.spec((u: any) => u.ifdirective((d: any) => d.condition(
      family.binary(">", family.identifier("N"), 1n)).items([enumerator("B", types)]).alternative((a: any) => a.elsedirective(
        (e: any) => e.items([enumerator("C", types)]))))), types.spec(comment(" end"))]);
    const sized = family.DIALECT.spec((u: any) => u.sizeofexpression((s: any) => s.operand(body)));
    assert(family.DIALECT.validate(term(sized, family.DIALECT)).length === 0);
    assert(printer(term(sized, family.DIALECT)) === "sizeof(enum E {\n    A\n#if N > 1\n    B,\n#else\n    C,\n#endif\n    // end\n})");
  }
  // A directive held from its own holder reads both languages: a C condition, a C++ item
  assert(validate((u: any) => u.ifdirective((d: any) => d.condition(condition).items([enumerator("A", XT.DIALECT)]))).length === 0);
}


<a id="CPD-06"></a>
## CPD-06 · Invalid syntax: error and missing among its terms, validated and printed as written, an error with no text refused by name

In [ ]:
{
  const [skipped, expected] = [(u: any) => u.error((r: any) => r.text("#if (")), (u: any) => u.missing((r: any) => r.kind("condition"))];
  assert(same(validate(skipped), ["invalid syntax '#if ('"]) && same(validate(expected), ["missing condition"]));
  assert(text(skipped) === "#if (" && text(expected) === "");
  assert(same(validate((u: any) => u.ifdirective((d: any) => d.condition((c: any) => c.c((v: any) => v.missing((m: any) => m.kind("expression")))))), [
    "condition: missing expression"]));
  raises(ValueError, () => text((u: any) => u.error((r: any) => r)), "C Preprocessor error has no text to print");
}


<a id="CPD-07"></a>
## CPD-07 · Its editions: C's three and C++'s three, each by its designation and common name, elifdef C23's and C++23's

In [ ]:
{
  const pre = store.Instances.Expressions.C.Preprocessor;
  assert(equal([...FS.standards(pre).values()].map((s) => FS.tags(s)), [
    ["ISO/IEC 9899:2011", "C11"], ["ISO/IEC 9899:2018", "C17"], ["ISO/IEC 9899:2024", "C23"], ["ISO/IEC 14882:2017", "C++17"],
    ["ISO/IEC 14882:2020", "C++20"], ["ISO/IEC 14882:2024", "C++23"]]) && same(PP.EDITIONS, ["C11", "C17", "C23", "C++17", "C++20", "C++23"]));
  assert(PP.SINCE.size === 1 && same(PP.SINCE.get("elifdefdirective")!, ["C23", "C++23"]));
  const kinds = schemas(PP.DIALECT.union);
  for (const [tag, schema] of kinds) {
    const expected = tag === "elifdefdirective" ? ["C23", "C++23"] : PP.TAGS.includes(tag) ? PP.EDITIONS : [];
    assert(same([...FS.of(pre, schema).values()].map((s) => FS.tags(s)[FS.tags(s).length - 1]), expected), tag);
  }
  assert(store.transaction(CPreprocessor.define)[0].get("Cpp23") === FS.find(pre, "C++23")); // run again, it writes nothing
  assert(Validators.Validate(store)(store.registered("Expressions.Framework.Standard"), FS.find(pre, "C11")).length === 0);
}


<a id="CPD-08"></a>
## CPD-08 · Printing: each kind as mbse-programs' printer writes it, a directive at the start of its line, a conditional one's branches on lines of their own

In [ ]:
{
  const cases: [unknown, string][] = [
    [(u: any) => u.includedirective((d: any) => d.path("a.h")), '#include "a.h"'],
    [(u: any) => u.includedirective((d: any) => d.path("stdio.h").system(true)), "#include <stdio.h>"],
    [(u: any) => u.includedirective((d: any) => d.directive("include_next").macro(C.identifier("HEADER"))), "#include_next HEADER"],
    [(u: any) => u.definedirective((d: any) => d.name(spelled("N"))), "#define N"],
    [(u: any) => u.definedirective((d: any) => d.name(spelled("N")).replacement("8")), "#define N 8"],
    [(u: any) => u.definedirective((d: any) => d.name(spelled("MAX")).function_like(true).parameters([spelled("a"), spelled("b")])
      .replacement("((a) > (b) ? (a) : (b))")), "#define MAX(a, b) ((a) > (b) ? (a) : (b))"],
    [(u: any) => u.definedirective((d: any) => d.name(spelled("LOG")).function_like(true).variadic(true).replacement("f(__VA_ARGS__)")),
      "#define LOG(...) f(__VA_ARGS__)"],
    [(u: any) => u.ifdefdirective((d: any) => d.name(spelled("A"))), "#ifdef A\n#endif"],
    [(u: any) => u.ifdefdirective((d: any) => d.negated(true).name(spelled("A")).items([enumerator("B")])), "#ifndef A\nB\n#endif"],
    [if_, "#if defined(X) && X > 1\nA\n#elifndef Z\nB\n#else\n// none\n#endif"],
    [(u: any) => u.ifdirective((d: any) => d.condition(C.literal(0n)).alternative((a: any) => a.elifdirective((e: any) => e.condition(
      CPP.binary("==", CPP.identifier("V"), 2n)).alternative((b: any) => b.elifdefdirective((f: any) => f.name(spelled("W"))))))),
      "#if 0\n#elif V == 2\n#elifdef W\n#endif"],
    [(u: any) => u.elsedirective((e: any) => e.items([enumerator("A")])), "#else\nA\n#endif"],
    [(u: any) => u.ifdirective((d: any) => d.condition(C.literal(1n)).items([enumerator("A"), own((c: any) => c.comment(
      (m: any) => m.text(" a").trailing(true)))])), "#if 1\nA // a\n#endif"],
    [(u: any) => u.otherdirective((d: any) => d.directive("pragma").text("once")), "#pragma once"],
    [(u: any) => u.otherdirective((d: any) => d.directive("undef").text("N")), "#undef N"],
    [(u: any) => u.otherdirective((d: any) => d), "#"], // the null directive
    [(u: any) => u.definedexpression((d: any) => d.name(spelled("X"))), "defined(X)"],
    [comment(" note"), "// note"], [comment(" a\n * b ", true), "/* a\n * b */"], [(u: any) => u.comment((c: any) => c), "//"],
  ];
  for (const [spec, source] of cases) assert(text(spec) === source, `${source} != ${text(spec)}`);
  raises(ValueError, () => text((u: any) => u.includedirective((d: any) => d)), "C Preprocessor includedirective has no path or macro to print");
}


<a id="CPD-09"></a>
## CPD-09 · Evaluation: none (Q11); each of its kinds is refused by name by its family's evaluators, wherever it stands

In [ ]:
{
  for (const spec of [(u: any) => u.includedirective((d: any) => d.path("a.h")), (u: any) => u.definedirective((d: any) => d.name(spelled("N"))),
    if_, (u: any) => u.otherdirective((d: any) => d.directive("pragma").text("once")), comment(" c"),
    (u: any) => u.definedexpression((d: any) => d.name(spelled("X")))]) {
    const tag = F.kindOf(term(spec))[0];
    raises(NotImplementedError, () => CV.OfAny(term(spec)), `C Preprocessor ${tag} is not evaluated`);
  }
  raises(NotImplementedError, () => CV.OfAny(term(C.binary("&&", defined("A"), 1n), C.DIALECT)), "C Preprocessor definedexpression is not evaluated"); // within C's expression
  raises(NotImplementedError, () => XV.OfAny(term(CPP.binary("||", defined("A", CPP.DIALECT), 0n), CPP.DIALECT)), "C Preprocessor definedexpression is not evaluated"); // and C++'s
}
